# Token Diffusion Model with Structure PCA Loss

This notebook implements a diffusion model that:
1. Uses pure x prediction (no continuous time or JVP)
2. Applies to images with conditioning tokens
3. Uses structure PCA loss (REPA loss) from semanticist repo
4. Trains with progressive token masking (all tokens -> 1 token)


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Configuration
class Config:
    # Core architecture
    img_size = 32
    patch_size = 4
    num_slots = 8           # Number of semantic tokens
    slot_dim = 64           # Dimension of semantic tokens
    
    # Feature configuration
    use_both_features = True    # True: use both image patches + semantic tokens
                               # False: use only semantic tokens (pure token diffusion)
    
    image_feature_dim = 8      # Dimensionality for image patch tokens (8x8 for 32x32 images with patch_size=4)
    
    # Training configuration
    num_timesteps = 1000       # Number of diffusion timesteps
    repa_weight = 0.1          # Weight for structure PCA loss
    learning_rate = 2e-4
    batch_size = 32
    
    # If using only token features, adjust timesteps to match token levels
    @property
    def effective_timesteps(self):
        if self.use_both_features:
            return self.num_timesteps
        else:
            # Pure token diffusion: timesteps = number of token masking levels
            return self.num_slots  # t=0 -> 8 tokens, t=7 -> 1 token
    
    def __repr__(self):
        return f"""Config:
  Architecture: {'Hybrid (patches + tokens)' if self.use_both_features else 'Pure token diffusion'}
  Image size: {self.img_size}x{self.img_size}, patch size: {self.patch_size}x{self.patch_size}
  Semantic tokens: {self.num_slots} x {self.slot_dim}D
  Image patches: {self.image_feature_dim}x{self.image_feature_dim} x {self.slot_dim}D
  Timesteps: {self.effective_timesteps} (original: {self.num_timesteps})
  REPA weight: {self.repa_weight}"""

config = Config()
print(config)

Using device: cuda
Config:
  Architecture: Hybrid (patches + tokens)
  Image size: 32x32, patch size: 4x4
  Semantic tokens: 8 x 64D
  Image patches: 8x8 x 64D
  Timesteps: 1000 (original: 1000)
  REPA weight: 0.1


## Vision Transformer Encoder for Token Generation

In [2]:
class VisionTokenizer(nn.Module):
    """Vision transformer that generates conditioning tokens with causal attention"""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.img_size = config.img_size
        self.patch_size = config.patch_size
        self.num_patches = (config.img_size // config.patch_size) ** 2
        self.num_slots = config.num_slots
        self.slot_dim = config.slot_dim
        self.embed_dim = 256  # Keep this fixed for now
        
        # Patch embedding
        self.patch_embed = nn.Conv2d(3, self.embed_dim, kernel_size=self.patch_size, stride=self.patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + self.num_slots + 1, self.embed_dim))
        
        # CLS token and slot tokens
        self.cls_token = nn.Parameter(torch.randn(1, 1, self.embed_dim))
        self.slot_tokens = nn.Parameter(torch.randn(1, self.num_slots, self.embed_dim))
        
        # Transformer layers
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=self.embed_dim, 
                nhead=8, 
                dim_feedforward=self.embed_dim*4,
                batch_first=True
            ), 
            num_layers=6
        )
        
        # Output projection for slots
        self.slot_proj = nn.Linear(self.embed_dim, self.slot_dim)
        
    def create_causal_mask(self, seq_len, device):
        """Create causal attention mask for slots"""
        mask = torch.ones(seq_len, seq_len, device=device, dtype=torch.bool)
        
        # Slots are at the end of sequence
        slots_start = seq_len - self.num_slots
        
        # Create causal mask for slots (lower triangular)
        causal_mask = torch.ones(self.num_slots, self.num_slots, device=device, dtype=torch.bool).tril(diagonal=0)
        mask[slots_start:, slots_start:] = causal_mask
        
        # CLS token and patches should not see slots
        mask[:slots_start, slots_start:] = False
        
        return ~mask  # Invert for attention mask
    
    def forward(self, x, is_causal=True):
        B = x.shape[0]
        
        # Patch embedding
        x = self.patch_embed(x)  # (B, embed_dim, H//patch_size, W//patch_size)
        x = x.flatten(2).transpose(1, 2)  # (B, num_patches, embed_dim)
        
        # Add cls token and slot tokens
        cls_tokens = self.cls_token.expand(B, -1, -1)
        slot_tokens = self.slot_tokens.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x, slot_tokens], dim=1)
        
        # Add positional embedding
        x = x + self.pos_embed
        
        # Apply causal attention mask if needed
        attn_mask = None
        if is_causal:
            attn_mask = self.create_causal_mask(x.shape[1], x.device)
        
        # Transformer
        x = self.transformer(x, mask=attn_mask)
        
        # Extract slot tokens
        slots = x[:, -self.num_slots:]
        slots = self.slot_proj(slots)
        
        return slots

## Simple Diffusion Model for X Prediction

In [3]:
class DiTBlock(nn.Module):
    """Transformer block with adaptive layer normalization for diffusion"""
    def __init__(self, hidden_size, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn = nn.MultiheadAttention(hidden_size, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        
        mlp_hidden_dim = int(hidden_size * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, hidden_size),
        )
        
        # Adaptive layer norm parameters (conditioned on timestep)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 6 * hidden_size, bias=True)
        )
    
    def forward(self, x, c):
        """
        x: input tokens [B, N, D]
        c: conditioning (time + other) [B, D]
        """
        # AdaLN modulation
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = \
            self.adaLN_modulation(c).chunk(6, dim=1)
        
        # Self-attention with AdaLN
        x_norm = self.norm1(x)
        x_norm = x_norm * (1 + scale_msa.unsqueeze(1)) + shift_msa.unsqueeze(1)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + gate_msa.unsqueeze(1) * attn_out
        
        # MLP with AdaLN
        x_norm = self.norm2(x)
        x_norm = x_norm * (1 + scale_mlp.unsqueeze(1)) + shift_mlp.unsqueeze(1)
        mlp_out = self.mlp(x_norm)
        x = x + gate_mlp.unsqueeze(1) * mlp_out
        
        return x


class DiffusionTransformer(nn.Module):
    """DiT for pure token-based diffusion"""
    def __init__(self, 
                 config,
                 hidden_size=512, 
                 depth=12, 
                 num_heads=8):
        super().__init__()
        self.config = config
        self.patch_size = config.patch_size
        self.img_size = config.img_size
        self.hidden_size = hidden_size
        self.num_patches = (config.img_size // config.patch_size) ** 2
        
        # Patch embedding for images
        self.x_embedder = nn.Linear(self.patch_size * self.patch_size * 3, hidden_size)
        
        # Time embedding
        self.t_embedder = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
        )
        
        # Conditioning token embedding - FIXED: create at init time
        self.c_embedder = nn.Linear(config.slot_dim, hidden_size)
        
        # FIXED: Semantic token projection - create at init time
        self.semantic_proj = nn.Linear(config.slot_dim, hidden_size)
        
        # Positional embedding
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, hidden_size))
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            DiTBlock(hidden_size, num_heads) for _ in range(depth)
        ])
        
        # Final layer
        self.final_layer = nn.Sequential(
            nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6),
            nn.Linear(hidden_size, self.patch_size * self.patch_size * 3, bias=True),
        )
        
        # AdaLN for final layer
        self.final_adaLN = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 2 * hidden_size, bias=True)
        )
        
        self.initialize_weights()
    
    def initialize_weights(self):
        # Initialize like ViT
        nn.init.xavier_uniform_(self.x_embedder.weight)
        nn.init.constant_(self.x_embedder.bias, 0)
        nn.init.normal_(self.pos_embed, std=0.02)
        
        # Zero-out final layer
        nn.init.constant_(self.final_layer[-1].weight, 0)
        nn.init.constant_(self.final_layer[-1].bias, 0)
    
    def patchify(self, imgs):
        """Convert images to patches: (B, C, H, W) -> (B, N, patch_size^2 * C)"""
        B, C, H, W = imgs.shape
        p = self.patch_size
        assert H == W == self.img_size
        
        h, w = H // p, W // p
        x = imgs.reshape(B, C, h, p, w, p)
        x = x.permute(0, 2, 4, 1, 3, 5).contiguous()
        x = x.reshape(B, h * w, p * p * C)
        return x
    
    def unpatchify(self, x):
        """Convert patches back to images: (B, N, patch_size^2 * C) -> (B, C, H, W)"""
        B, N, _ = x.shape
        p = self.patch_size
        c = 3
        h = w = int(N ** 0.5)
        assert h * w == N
        
        x = x.reshape(B, h, w, c, p, p)
        x = x.permute(0, 3, 1, 4, 2, 5).contiguous()
        x = x.reshape(B, c, h * p, w * p)
        return x
    
    def forward(self, x, t, conditioning_tokens):
        """
        x: noisy images [B, 3, H, W] or None for pure token mode
        t: timesteps [B]
        conditioning_tokens: semantic tokens [B, num_slots, conditioning_dim]
        """
        B = t.shape[0]
        
        # Time embedding
        t_emb = self.t_embedder(t.float().unsqueeze(-1) / self.config.effective_timesteps)
        
        # Conditioning embedding (mean pool semantic tokens)
        c_emb = conditioning_tokens.mean(dim=1)  # [B, conditioning_dim]
        c_emb = self.c_embedder(c_emb)
        
        # Combined conditioning
        c = t_emb + c_emb  # [B, hidden_size]
        
        if self.config.use_both_features and x is not None:
            # Hybrid mode: use both image patches and semantic tokens
            # Patchify and embed images
            x_patches = self.patchify(x)  # [B, N, patch_size^2 * 3]
            x_tokens = self.x_embedder(x_patches) + self.pos_embed
            
            # Add semantic tokens (use pre-created projection layer)
            semantic_tokens = self.semantic_proj(conditioning_tokens)
            
            # Combine image patches + semantic tokens
            tokens = torch.cat([x_tokens, semantic_tokens], dim=1)  # [B, N+num_slots, hidden_size]
        else:
            # Pure token mode: only semantic tokens
            tokens = self.semantic_proj(conditioning_tokens)  # [B, num_slots, hidden_size]
        
        # Apply transformer blocks
        for block in self.blocks:
            tokens = block(tokens, c)
        
        if self.config.use_both_features and x is not None:
            # Extract image patch tokens (first N tokens)
            image_tokens = tokens[:, :self.num_patches]
            
            # Final layer with AdaLN
            shift, scale = self.final_adaLN(c).chunk(2, dim=1)
            image_tokens = self.final_layer[0](image_tokens)  # LayerNorm
            image_tokens = image_tokens * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)
            image_tokens = self.final_layer[1](image_tokens)  # Linear
            
            # Convert back to image
            x_pred = self.unpatchify(image_tokens)
            return x_pred
        else:
            # Pure token mode: return processed semantic tokens
            return tokens

## Progressive Token Masking Tied to Diffusion Timesteps

**KEY INSIGHT**: The number of active tokens is now directly tied to the diffusion timestep `t`!

This creates a **dual noise schedule**:
1. **Image noise**: `x_t = √(α_t) * x_0 + √(1-α_t) * ε` (continuous Gaussian)
2. **Token noise**: `n_active = f(t)` where fewer tokens = more "semantic noise"

**Training Schedule**:
- `t=0` (clean image) → `n_active=8` tokens (full semantic info)
- `t=250` → `n_active=6` tokens  
- `t=500` → `n_active=4` tokens
- `t=750` → `n_active=2` tokens
- `t=999` (noisy image) → `n_active=1` token (minimal semantic info)

The model learns to predict clean images from both noisy pixels AND degraded semantic tokens!

In [4]:
class ProgressiveTokenMask(nn.Module):
    """Implements progressive token masking from all tokens to 1 token"""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_slots = config.num_slots
        self.null_token = nn.Parameter(torch.randn(1, 1, config.slot_dim))  # Learnable null token
    
    def uniform_sample(self, batch_size, device):
        """Sample number of active tokens uniformly from [1, num_slots]"""
        return torch.randint(1, self.num_slots + 1, (batch_size,), device=device)
    
    def tokens_from_timestep(self, t, num_timesteps):
        """Convert diffusion timestep to number of active tokens
        
        For hybrid mode (use_both_features=True):
        t=0 (clean) -> num_slots tokens (full semantic information)
        t=num_timesteps-1 (noisy) -> 1 token (minimal semantic information)
        
        For pure token mode (use_both_features=False):
        t=0 -> num_slots tokens, t=num_slots-1 -> 1 token
        """
        if self.config.use_both_features:
            # Standard diffusion timesteps (1000 steps)
            progress = t.float() / (num_timesteps - 1)  # 0 to 1
        else:
            # Pure token mode: timesteps = token levels
            progress = t.float() / max(1, num_timesteps - 1)  # 0 to 1
        
        n_active = self.num_slots - (progress * (self.num_slots - 1))  # num_slots to 1
        n_active = torch.round(n_active).long().clamp(1, self.num_slots)
        return n_active
    
    def create_mask(self, batch_size, device, inference_n_slots=None, t=None, num_timesteps=None):
        """Create mask for active tokens"""
        if inference_n_slots is not None:
            # For inference, use specified number of slots
            n_active = torch.full((batch_size,), inference_n_slots, device=device)
        elif t is not None and num_timesteps is not None:
            # For training, derive from timestep
            n_active = self.tokens_from_timestep(t, num_timesteps)
        else:
            # Fallback: sample randomly (old behavior)
            n_active = self.uniform_sample(batch_size, device)
        
        # Create mask: first n_active tokens are True, rest are False
        arange = torch.arange(self.num_slots, device=device)[None, :]
        mask = arange < n_active[:, None]
        
        return mask, n_active
    
    def forward(self, tokens, mask):
        """Apply mask to tokens, replacing masked tokens with null token"""
        batch_size, num_tokens, token_dim = tokens.shape
        null_tokens = self.null_token.expand(batch_size, num_tokens, -1)
        
        # Apply mask
        masked_tokens = torch.where(mask[:, :, None], tokens, null_tokens)
        
        return masked_tokens

## Structure PCA Loss (REPA Loss)

In [5]:
class REPALoss(nn.Module):
    """REPA loss implementation based on semanticist repo"""
    def __init__(self, config, feature_dim=512):
        super().__init__()
        self.config = config
        # Pre-trained feature encoder (simplified - in practice use DINOv2)
        self.feature_encoder = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, feature_dim)
        )
        
        # Freeze pre-trained encoder
        for param in self.feature_encoder.parameters():
            param.requires_grad = False
    
    def interpolate_features(self, x, target_len):
        """Interpolate features to match target sequence length"""
        B, T1, D = x.shape
        if T1 == target_len:
            return x
        
        H1 = W1 = int(math.sqrt(T1))
        H2 = W2 = int(math.sqrt(target_len))
        
        # Reshape to 2D spatial and interpolate
        x = x.reshape(B, H1, W1, D).permute(0, 3, 1, 2)
        x = F.interpolate(x, size=(H2, W2), mode='bicubic', align_corners=False)
        return x.permute(0, 2, 3, 1).reshape(B, target_len, D)
    
    def forward(self, tokens, images):
        """Compute REPA loss between tokens and image features"""
        # Get pre-trained features
        with torch.no_grad():
            img_features = self.feature_encoder(images)  # (B, feature_dim)
            img_features = img_features.unsqueeze(1)  # (B, 1, feature_dim)
        
        # Token features (mean pooling)
        token_features = tokens.mean(dim=1, keepdim=True)  # (B, 1, token_dim)
        
        # Project to same dimension if needed
        if token_features.shape[-1] != img_features.shape[-1]:
            token_features = F.interpolate(
                token_features.transpose(1, 2), 
                size=img_features.shape[-1], 
                mode='linear'
            ).transpose(1, 2)
        
        # L2 normalize
        token_features = F.normalize(token_features, dim=-1)
        img_features = F.normalize(img_features, dim=-1)
        
        # Negative cosine similarity
        repa_loss = -torch.sum(token_features * img_features, dim=-1)
        
        return repa_loss.mean()

## Main Training Model

In [6]:
class TokenDiffusionModel(nn.Module):
    """Main model combining tokenizer, DiT, and losses"""
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # Components - ALL now receive config
        self.tokenizer = VisionTokenizer(config)
        
        self.diffusion_model = DiffusionTransformer(
            config,
            hidden_size=512,
            depth=8,  # Smaller for testing
            num_heads=8
        )
        
        self.token_mask = ProgressiveTokenMask(config)
        self.repa_loss = REPALoss(config)
        
        # Noise schedule (only for hybrid mode)
        if config.use_both_features:
            betas = torch.linspace(0.0001, 0.02, config.num_timesteps)
            alphas = 1.0 - betas
            alphas_cumprod = torch.cumprod(alphas, dim=0)
            
            self.register_buffer('betas', betas)
            self.register_buffer('alphas', alphas)
            self.register_buffer('alphas_cumprod', alphas_cumprod)
            self.register_buffer('sqrt_alphas_cumprod', torch.sqrt(alphas_cumprod))
            self.register_buffer('sqrt_one_minus_alphas_cumprod', torch.sqrt(1.0 - alphas_cumprod))
        else:
            # Pure token mode: no image noise schedule needed
            self.register_buffer('betas', None)
    
    def add_noise(self, x_0, t):
        """Add noise to clean images according to noise schedule (only for hybrid mode)"""
        if not self.config.use_both_features:
            # Pure token mode: no image noise
            return x_0, torch.zeros_like(x_0)
            
        noise = torch.randn_like(x_0)
        sqrt_alphas_cumprod_t = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        
        x_t = sqrt_alphas_cumprod_t * x_0 + sqrt_one_minus_alphas_cumprod_t * noise
        return x_t, noise
    
    def forward(self, images):
        """Forward pass for training"""
        batch_size = images.shape[0]
        device = images.device
        
        # Generate tokens with causal attention
        tokens = self.tokenizer(images, is_causal=True)
        
        # Sample timesteps (different logic for pure token vs hybrid)
        if self.config.use_both_features:
            t = torch.randint(0, self.config.num_timesteps, (batch_size,), device=device)
        else:
            # Pure token mode: t corresponds to token masking levels
            t = torch.randint(0, self.config.num_slots, (batch_size,), device=device)
        
        # Progressive token masking BASED ON TIMESTEP t
        mask, n_active = self.token_mask.create_mask(
            batch_size, device, t=t, num_timesteps=self.config.effective_timesteps
        )
        masked_tokens = self.token_mask(tokens, mask)
        
        if self.config.use_both_features:
            # Hybrid mode: add noise to images
            x_t, noise = self.add_noise(images, t)
            x_pred = self.diffusion_model(x_t, t, masked_tokens)
            
            # EXPLICIT DIFFUSION LOSS: ||x_0 - x_pred||²
            diff_loss = torch.mean((images - x_pred) ** 2)
        else:
            # Pure token mode: only token-based diffusion
            # The DiT processes semantic tokens only and outputs processed tokens
            processed_tokens = self.diffusion_model(None, t, masked_tokens)  # No image input
            
            # Loss: reconstruct original semantic tokens
            diff_loss = torch.mean((tokens - processed_tokens) ** 2)
            x_pred = None  # No image prediction in pure token mode
        
        # REPA loss (structure PCA loss)
        repa_loss = self.repa_loss(tokens, images)
        
        # Total loss
        total_loss = diff_loss + self.config.repa_weight * repa_loss
        
        result = {
            'total_loss': total_loss,
            'diff_loss': diff_loss,
            'repa_loss': repa_loss,
            'n_active_tokens': n_active.float().mean(),
            'timestep_mean': t.float().mean().item(),
            'mode': 'hybrid' if self.config.use_both_features else 'pure_token'
        }
        
        if x_pred is not None:
            result.update({
                'x_pred_mean': x_pred.mean().item(),
                'x_0_mean': images.mean().item()
            })
        
        return result
    
    @torch.no_grad()
    def sample(self, batch_size, n_slots=None, num_steps=50):
        """Sample images using DDIM-like sampling"""
        device = next(self.parameters()).device
        
        if n_slots is None:
            n_slots = self.config.num_slots
        
        if self.config.use_both_features:
            # Hybrid mode: start with noisy images + tokens
            x = torch.randn(batch_size, 3, self.config.img_size, self.config.img_size, device=device)
            timesteps = torch.linspace(self.config.num_timesteps-1, 0, num_steps, dtype=torch.long, device=device)
        else:
            # Pure token mode: start with random tokens
            x = None
            timesteps = torch.linspace(self.config.num_slots-1, 0, num_steps, dtype=torch.long, device=device)
        
        # Create conditioning tokens
        if self.config.use_both_features:
            dummy_image = torch.zeros(batch_size, 3, self.config.img_size, self.config.img_size, device=device)
        else:
            dummy_image = torch.randn(batch_size, 3, self.config.img_size, self.config.img_size, device=device)
        
        tokens = self.tokenizer(dummy_image, is_causal=True)
        
        for i, t in enumerate(timesteps):
            t_batch = t.expand(batch_size)
            
            # Create mask for current timestep
            mask, _ = self.token_mask.create_mask(
                batch_size, device, t=t_batch, num_timesteps=self.config.effective_timesteps
            )
            masked_tokens = self.token_mask(tokens, mask)
            
            if self.config.use_both_features:
                # Predict original image
                x_pred = self.diffusion_model(x, t_batch, masked_tokens)
                
                if i < len(timesteps) - 1:
                    # Not the last step, add some noise
                    alpha_t = self.alphas_cumprod[t]
                    alpha_t_prev = self.alphas_cumprod[timesteps[i+1]]
                    
                    # DDIM update
                    noise = torch.randn_like(x) if i < len(timesteps) - 1 else torch.zeros_like(x)
                    x = torch.sqrt(alpha_t_prev) * x_pred + torch.sqrt(1 - alpha_t_prev) * noise
                else:
                    x = x_pred
                    
                final_result = torch.clamp(x, -1, 1)
            else:
                # Pure token mode: process tokens iteratively
                processed_tokens = self.diffusion_model(None, t_batch, masked_tokens)
                
                if i == len(timesteps) - 1:
                    # Final step: decode tokens to image (would need a decoder)
                    # For now, just return a dummy image
                    final_result = torch.randn(batch_size, 3, self.config.img_size, self.config.img_size, device=device)
                else:
                    tokens = processed_tokens
        
        return final_result

## Data Loading and Training

In [7]:
# Data loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

# Use CIFAR-10 for minimal example
dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=config.batch_size, shuffle=True, num_workers=2)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(dataloader)}")
print(f"Batch size: {config.batch_size}")

Dataset size: 50000
Number of batches: 1563
Batch size: 32


In [8]:
# Quick test to verify the model works
print("🧪 Testing model initialization and forward pass...")

try:
    # Test model creation - NOW PASSES CONFIG
    model = TokenDiffusionModel(config).to(device)
    print(f"✅ Model created successfully with {sum(p.numel() for p in model.parameters()):,} parameters")
    
    # Test forward pass with dummy data
    test_images = torch.randn(4, 3, config.img_size, config.img_size).to(device)
    
    with torch.no_grad():
        model.eval()
        result = model(test_images)
    
    print(f"✅ Forward pass successful!")
    print(f"   Mode: {result['mode']}")
    print(f"   Loss components: {list(result.keys())}")
    print(f"   Active tokens: {result['n_active_tokens']:.1f}")
    print(f"   Timestep: {result['timestep_mean']:.1f}")
    
    # Test sampling
    print("\n🎨 Testing sampling...")
    samples = model.sample(batch_size=2, n_slots=4, num_steps=5)  # Quick test
    print(f"✅ Sampling successful! Generated images shape: {samples.shape}")
    
    print("\n🚀 All tests passed! The model is ready to train.")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

🧪 Testing model initialization and forward pass...
✅ Model created successfully with 43,579,568 parameters
❌ Error: The size of tensor a (64) must match the size of tensor b (512) at non-singleton dimension 2


Traceback (most recent call last):
  File "/tmp/ipykernel_8574/3065477166.py", line 14, in <module>
    result = model(test_images)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1750, in _call_impl
    return forward_call(*args, **kwargs)
  File "/tmp/ipykernel_8574/2244247994.py", line 86, in forward
    repa_loss = self.repa_loss(tokens, images)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1750, in _call_impl
    return forward_call(*args, **kwargs)
  File "/tmp/ipykernel_8574/2634487495.py", line 56, in

In [9]:
# Initialize model for training
print("🚀 Initializing training setup...")

# FIXED: Pass config to model
model = TokenDiffusionModel(config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Configuration: {'Hybrid' if config.use_both_features else 'Pure Token'}")
print(f"Effective timesteps: {config.effective_timesteps}")

# Test different configurations
print("\n" + "="*50)
print("CONFIGURATION DETAILS:")
print("="*50)

print(f"\nCurrent config (Hybrid: {config.use_both_features}):")
print(f"  - Timesteps: {config.effective_timesteps}")
print(f"  - Image features: {'Yes' if config.use_both_features else 'No'}")
print(f"  - Token progression: {config.num_slots} -> 1 tokens")

if config.use_both_features:
    print(f"\nTo switch to pure token mode, change:")
    print(f"  config.use_both_features = False")
    print(f"  This would give:")
    print(f"  - Timesteps: {config.num_slots} (same as token levels)")
    print(f"  - Image features: No")
    print(f"  - Token progression: {config.num_slots} -> 1 tokens")
    print(f"  - Loss: ||tokens_original - tokens_predicted||²")

🚀 Initializing training setup...
Model parameters: 43,579,568
Configuration: Hybrid
Effective timesteps: 1000

CONFIGURATION DETAILS:

Current config (Hybrid: True):
  - Timesteps: 1000
  - Image features: Yes
  - Token progression: 8 -> 1 tokens

To switch to pure token mode, change:
  config.use_both_features = False
  This would give:
  - Timesteps: 8 (same as token levels)
  - Image features: No
  - Token progression: 8 -> 1 tokens
  - Loss: ||tokens_original - tokens_predicted||²


In [10]:
# Train for a few epochs
num_epochs = 3

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    # Train
    train_stats = train_epoch(model, dataloader, optimizer, device)
    
    print(f"Train - Total: {train_stats['total_loss']:.4f}, "
          f"Diff: {train_stats['diff_loss']:.4f}, "
          f"REPA: {train_stats['repa_loss']:.4f}, "
          f"Avg Tokens: {train_stats['avg_tokens']:.1f}")
    
    # Sample some images
    if (epoch + 1) % 2 == 0:
        print("Sampling images...")
        samples = model.sample(batch_size=8, n_slots=4, num_steps=20)
        samples = (samples + 1) / 2  # Denormalize to [0, 1]
        
        # Plot samples
        fig, axes = plt.subplots(2, 4, figsize=(12, 6))
        for i in range(8):
            ax = axes[i // 4, i % 4]
            ax.imshow(samples[i].permute(1, 2, 0).cpu().numpy())
            ax.axis('off')
        plt.suptitle(f"Generated Images - Epoch {epoch+1}")
        plt.tight_layout()
        plt.show()


Epoch 1/3


NameError: name 'train_epoch' is not defined

## Test Progressive Token Masking

## Reconstruction Testing

This section tests the model's ability to reconstruct images from varying numbers of structured tokens, which is the core capability we're training for.

def test_reconstruction(model, test_images, n_structured_tokens, use_patch_tokens=True, timestep=None):
    """
    Test reconstruction capability of the model
    
    Args:
        model: trained model
        test_images: batch of test images [B, 3, H, W]
        n_structured_tokens: number of structured tokens to use (1 to config.num_slots)
        use_patch_tokens: if True, use noisy patch tokens; if False, use noise
        timestep: specific timestep to test (if None, uses t=0 for clean reconstruction)
    """
    model.eval()
    device = test_images.device
    batch_size = test_images.shape[0]
    
    with torch.no_grad():
        # Generate structured tokens from clean images
        structured_tokens = model.tokenizer(test_images, is_causal=True)
        
        # Create mask for the desired number of tokens
        mask, _ = model.token_mask.create_mask(
            batch_size, device, inference_n_slots=n_structured_tokens
        )
        masked_tokens = model.token_mask(structured_tokens, mask)
        
        if timestep is None:
            timestep = 0  # Clean reconstruction
        
        t = torch.full((batch_size,), timestep, device=device)
        
        if config.use_both_features and use_patch_tokens:
            # Use noisy patch tokens + structured tokens
            if timestep == 0:
                # Clean patches for t=0
                x_input = test_images
            else:
                # Add noise according to timestep
                x_input, _ = model.add_noise(test_images, t)
            
            # Reconstruct using DiT
            reconstructed = model.diffusion_model(x_input, t, masked_tokens)
        
        elif config.use_both_features and not use_patch_tokens:
            # Use noise instead of patch tokens + structured tokens
            noise_input = torch.randn_like(test_images)
            reconstructed = model.diffusion_model(noise_input, t, masked_tokens)
        
        else:
            # Pure token mode: only structured tokens (no image patches)
            # This would need a separate decoder to convert tokens back to images
            # For now, we'll create a simple mapping
            processed_tokens = model.diffusion_model(None, t, masked_tokens)
            
            # Simple token-to-image mapping (this would be learned in practice)
            # Just return a placeholder reconstruction
            reconstructed = torch.tanh(torch.randn_like(test_images))
    
    return reconstructed, masked_tokens, structured_tokens


def calculate_reconstruction_metrics(original, reconstructed):
    """Calculate reconstruction quality metrics"""
    # MSE
    mse = torch.mean((original - reconstructed) ** 2)
    
    # PSNR
    psnr = 20 * torch.log10(2.0) - 10 * torch.log10(mse)  # Range is [-1, 1]
    
    # SSIM (simplified version)
    def ssim_1d(x, y):
        mu_x = torch.mean(x)
        mu_y = torch.mean(y)
        sigma_x = torch.var(x)
        sigma_y = torch.var(y)
        sigma_xy = torch.mean((x - mu_x) * (y - mu_y))
        
        c1, c2 = 0.01, 0.03
        ssim = ((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) / \
               ((mu_x**2 + mu_y**2 + c1) * (sigma_x + sigma_y + c2))
        return ssim
    
    # Average SSIM across channels and spatial dimensions
    original_flat = original.view(original.shape[0], -1)
    reconstructed_flat = reconstructed.view(reconstructed.shape[0], -1)
    
    ssim_scores = []
    for i in range(original.shape[0]):
        ssim = ssim_1d(original_flat[i], reconstructed_flat[i])
        ssim_scores.append(ssim.item())
    
    avg_ssim = sum(ssim_scores) / len(ssim_scores)
    
    return {
        'mse': mse.item(),
        'psnr': psnr.item(), 
        'ssim': avg_ssim
    }


def comprehensive_reconstruction_test(model, dataloader, num_test_images=8):
    """Run comprehensive reconstruction tests"""
    print("🔬 Running Comprehensive Reconstruction Tests")
    print("=" * 60)
    
    # Get test images from both train and test sets
    test_images = []
    with torch.no_grad():
        for i, (images, _) in enumerate(dataloader):
            test_images.append(images[:4])  # Take 4 from each batch
            if len(test_images) >= 2:  # Get 8 total images
                break
    
    test_images = torch.cat(test_images, dim=0)[:num_test_images].to(device)
    
    # Test configurations
    token_counts = [1, 2, 4, 8]
    patch_configurations = [
        ("Clean Patches", True, 0),
        ("Noisy Patches (t=250)", True, 250 if config.use_both_features else 2),
        ("Noisy Patches (t=500)", True, 500 if config.use_both_features else 4), 
        ("No Patches (Noise)", False, 0)
    ]
    
    results = {}
    
    for n_tokens in token_counts:
        print(f"\n📊 Testing with {n_tokens} structured tokens:")
        print("-" * 40)
        
        results[n_tokens] = {}
        
        for config_name, use_patches, timestep in patch_configurations:
            if not config.use_both_features and use_patches and timestep > 0:
                continue  # Skip noisy patches in pure token mode
            
            try:
                reconstructed, masked_tokens, original_tokens = test_reconstruction(
                    model, test_images, n_tokens, use_patches, timestep
                )
                
                metrics = calculate_reconstruction_metrics(test_images, reconstructed)
                results[n_tokens][config_name] = {
                    'metrics': metrics,
                    'reconstructed': reconstructed,
                    'masked_tokens': masked_tokens
                }
                
                print(f"  {config_name:20} | "
                      f"MSE: {metrics['mse']:.4f} | "
                      f"PSNR: {metrics['psnr']:.2f} dB | "
                      f"SSIM: {metrics['ssim']:.3f}")
                
            except Exception as e:
                print(f"  {config_name:20} | Error: {str(e)}")
                results[n_tokens][config_name] = {'error': str(e)}
    
    return results, test_images


def visualize_reconstruction_results(results, original_images, save_path=None):
    """Visualize reconstruction results"""
    token_counts = list(results.keys())
    config_names = list(results[token_counts[0]].keys())
    
    # Filter out configurations with errors
    valid_configs = [name for name in config_names if 'error' not in results[token_counts[0]][name]]
    
    if not valid_configs:
        print("No valid configurations to visualize")
        return
    
    # Create visualization grid
    num_images = min(4, len(original_images))
    fig, axes = plt.subplots(
        len(token_counts) + 1, 
        len(valid_configs) + 1,  # +1 for original images
        figsize=(4 * (len(valid_configs) + 1), 4 * (len(token_counts) + 1))
    )
    
    if len(axes.shape) == 1:
        axes = axes.reshape(-1, 1)
    
    # Show original images in first row
    for j in range(len(valid_configs) + 1):
        if j == 0:
            axes[0, j].text(0.5, 0.5, 'Original\nImages', ha='center', va='center', 
                           transform=axes[0, j].transAxes, fontsize=12, weight='bold')
            axes[0, j].axis('off')
        else:
            config_name = valid_configs[j-1]
            axes[0, j].text(0.5, 0.5, config_name, ha='center', va='center',
                           transform=axes[0, j].transAxes, fontsize=10, weight='bold')
            axes[0, j].axis('off')
    
    # Show reconstructions for each token count
    for i, n_tokens in enumerate(token_counts):
        row = i + 1
        
        # Label row
        axes[row, 0].text(0.5, 0.5, f'{n_tokens} Tokens', ha='center', va='center',
                         transform=axes[row, 0].transAxes, fontsize=12, weight='bold', rotation=90)
        axes[row, 0].axis('off')
        
        # Show reconstructions for each configuration
        for j, config_name in enumerate(valid_configs):
            col = j + 1
            
            if 'error' in results[n_tokens][config_name]:
                axes[row, col].text(0.5, 0.5, 'Error', ha='center', va='center',
                                   transform=axes[row, col].transAxes, color='red')
                axes[row, col].axis('off')
                continue
            
            # Create a grid of reconstructed images
            reconstructed = results[n_tokens][config_name]['reconstructed']
            
            # Show first image from batch
            img = reconstructed[0].cpu()
            img = torch.clamp((img + 1) / 2, 0, 1)  # Denormalize to [0, 1]
            
            axes[row, col].imshow(img.permute(1, 2, 0).numpy())
            axes[row, col].axis('off')
            
            # Add metrics as subtitle
            metrics = results[n_tokens][config_name]['metrics']
            axes[row, col].set_title(
                f"MSE: {metrics['mse']:.3f}\n"
                f"PSNR: {metrics['psnr']:.1f} dB\n"
                f"SSIM: {metrics['ssim']:.3f}",
                fontsize=8
            )
    
    # Add original images in a separate subplot for comparison
    fig2, axes2 = plt.subplots(1, num_images, figsize=(3*num_images, 3))
    if num_images == 1:
        axes2 = [axes2]
    
    for i in range(num_images):
        img = original_images[i].cpu()
        img = torch.clamp((img + 1) / 2, 0, 1)  # Denormalize
        axes2[i].imshow(img.permute(1, 2, 0).numpy())
        axes2[i].set_title(f'Original {i+1}')
        axes2[i].axis('off')
    
    plt.figure(fig.number)
    plt.suptitle('Reconstruction Quality vs. Number of Structured Tokens', fontsize=16, y=0.98)
    plt.tight_layout()
    
    plt.figure(fig2.number) 
    plt.suptitle('Original Test Images', fontsize=14)
    plt.tight_layout()
    
    if save_path:
        fig.savefig(f"{save_path}_reconstruction_grid.png", dpi=150, bbox_inches='tight')
        fig2.savefig(f"{save_path}_originals.png", dpi=150, bbox_inches='tight')
    
    plt.show()
    
    return fig, fig2

In [ ]:
# Run reconstruction tests after training
print("🧪 RECONSTRUCTION CAPABILITY TESTING")
print("=" * 60)
print("This tests how well the model can reconstruct images from different numbers")
print("of structured tokens, which is the core capability we're training.")
print()

# Get test dataset for evaluation
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2)

print(f"Test dataset size: {len(test_dataset)}")
print(f"Using configuration: {'Hybrid' if config.use_both_features else 'Pure Token'}")
print()

# Run comprehensive tests
results, test_images = comprehensive_reconstruction_test(model, test_dataloader, num_test_images=8)

print("\n" + "=" * 60)
print("📈 RECONSTRUCTION QUALITY SUMMARY")
print("=" * 60)

# Summary statistics
token_counts = sorted(results.keys())
configurations = list(results[token_counts[0]].keys())

print(f"\n{'Tokens':<8} | {'Configuration':<20} | {'MSE':<8} | {'PSNR':<8} | {'SSIM':<8}")
print("-" * 70)

best_results = {}
for n_tokens in token_counts:
    best_results[n_tokens] = {'best_mse': float('inf'), 'best_config': None}
    
    for config_name in configurations:
        if 'error' not in results[n_tokens][config_name]:
            metrics = results[n_tokens][config_name]['metrics']
            mse = metrics['mse']
            
            print(f"{n_tokens:<8} | {config_name:<20} | "
                  f"{mse:<8.4f} | {metrics['psnr']:<8.2f} | {metrics['ssim']:<8.3f}")
            
            if mse < best_results[n_tokens]['best_mse']:
                best_results[n_tokens]['best_mse'] = mse
                best_results[n_tokens]['best_config'] = config_name

print("\n🏆 BEST RECONSTRUCTION FOR EACH TOKEN COUNT:")
print("-" * 50)
for n_tokens in token_counts:
    best_config = best_results[n_tokens]['best_config']
    if best_config:
        best_metrics = results[n_tokens][best_config]['metrics']
        print(f"{n_tokens} tokens: {best_config} (MSE: {best_metrics['mse']:.4f}, "
              f"PSNR: {best_metrics['psnr']:.2f} dB, SSIM: {best_metrics['ssim']:.3f})")

print("\n📊 INSIGHTS:")
print("-" * 20)
print("• Lower MSE = Better reconstruction quality")
print("• Higher PSNR = Better signal-to-noise ratio")
print("• Higher SSIM = Better structural similarity")
print("• Look for how quality degrades as token count decreases")
print("• This shows the effectiveness of the front-loaded token approach")

# Visualize results
print("\n🖼️  Generating visualization...")
visualize_reconstruction_results(results, test_images, save_path="reconstruction_test")

# Additional test: Compare reconstruction on training vs test data
print("\n" + "="*60)
print("🔄 TRAINING vs TEST SET RECONSTRUCTION COMPARISON") 
print("="*60)
print("This compares how well the model reconstructs training vs test images")
print("to check for overfitting in the reconstruction capability.")
print()

def compare_train_test_reconstruction(model, train_loader, test_loader, n_tokens=4):
    """Compare reconstruction quality on train vs test data"""
    print(f"🔍 Testing reconstruction with {n_tokens} structured tokens...")
    
    results_comparison = {'train': {}, 'test': {}}
    
    # Get samples from training set
    train_images = []
    for i, (images, _) in enumerate(train_loader):
        train_images.append(images[:2])
        if len(train_images) >= 2:
            break
    train_images = torch.cat(train_images, dim=0)[:4].to(device)
    
    # Get samples from test set  
    test_images = []
    for i, (images, _) in enumerate(test_loader):
        test_images.append(images[:2])
        if len(test_images) >= 2:
            break
    test_images = torch.cat(test_images, dim=0)[:4].to(device)
    
    # Test different configurations on both sets
    configurations = [
        ("Clean Patches", True, 0),
        ("No Patches (Noise)", False, 0)
    ]
    
    for dataset_name, images in [("train", train_images), ("test", test_images)]:
        print(f"\n📊 {dataset_name.upper()} SET:")
        results_comparison[dataset_name] = {}
        
        for config_name, use_patches, timestep in configurations:
            try:
                reconstructed, _, _ = test_reconstruction(model, images, n_tokens, use_patches, timestep)
                metrics = calculate_reconstruction_metrics(images, reconstructed)
                results_comparison[dataset_name][config_name] = metrics
                
                print(f"  {config_name:18} | MSE: {metrics['mse']:.4f} | "
                      f"PSNR: {metrics['psnr']:.2f} dB | SSIM: {metrics['ssim']:.3f}")
                
            except Exception as e:
                print(f"  {config_name:18} | Error: {str(e)}")
    
    # Calculate differences
    print(f"\n📈 TRAIN vs TEST DIFFERENCES:")
    print("-" * 40)
    
    for config_name in configurations:
        config_name = config_name[0]
        if config_name in results_comparison['train'] and config_name in results_comparison['test']:
            train_mse = results_comparison['train'][config_name]['mse'] 
            test_mse = results_comparison['test'][config_name]['mse']
            
            diff = test_mse - train_mse
            diff_pct = (diff / train_mse) * 100
            
            print(f"{config_name:18} | Train MSE: {train_mse:.4f} | "
                  f"Test MSE: {test_mse:.4f} | Diff: {diff:+.4f} ({diff_pct:+.1f}%)")
    
    print(f"\n💡 INTERPRETATION:")
    print("• If test MSE >> train MSE: possible overfitting")  
    print("• If test MSE ≈ train MSE: good generalization")
    print("• Small differences are expected and normal")
    
    return results_comparison

# Run the comparison
comparison_results = compare_train_test_reconstruction(model, dataloader, test_dataloader, n_tokens=4)